In [1]:
import matlab.engine
import json
from pathlib import Path

In [2]:
eng = matlab.engine.start_matlab()
model_path = "Matlab/P1"
eng.load_system(model_path, nargout=0)

In [4]:
eng.workspace["Kp"] = 10.0
eng.workspace["Ki"] = 10.0
eng.workspace["Kd"] = 1.0
eng.workspace["m"] = 1.0
eng.workspace["c"] = 2.0
eng.workspace["k"] = 5.0

In [5]:
sim_out = eng.sim(model_path, nargout=1)

In [6]:
eng.eval(f"out = sim('{model_path}');", nargout=0)

In [7]:
pos_out = eng.eval("out.pos_out", nargout=1)
vel_out = eng.eval("out.vel_out", nargout=1)
t_out   = eng.eval("out.t_out", nargout=1)

In [9]:
print(type(pos_out))
print(type(vel_out))
print(type(t_out))

<class 'matlab.double'>
<class 'matlab.double'>
<class 'matlab.double'>


In [10]:
print(eng.eval("class(out.pos_out)", nargout=1))
print(eng.eval("class(out.vel_out)", nargout=1))
print(eng.eval("class(out.t_out)", nargout=1))

double
double
double


In [13]:
eng.eval("info_pos = stepinfo(out.pos_out(:), out.t_out(:));", nargout=0)
print(eng.eval("info_pos", nargout=1))

{'RiseTime': 0.46223587287315143, 'TransientTime': 3.941383441106845, 'SettlingTime': 3.941383441106845, 'SettlingMin': 0.8141643016908532, 'SettlingMax': 1.076869467168402, 'Overshoot': 7.705394514974873, 'Undershoot': -0.0, 'Peak': 1.076869467168402, 'PeakTime': 0.8392230080215957}


In [15]:
# RiseTime:
# 출력이 초기값에서 최종값 근처까지 처음 도달하는 데 걸리는 시간
# 보통 응답의 "초기 상승 속도"를 보는 지표
print("RiseTime:", eng.eval("info_pos.RiseTime", nargout=1))

# TransientTime:
# 과도응답이 사실상 끝나는 시간
# MATLAB에서는 settling과 유사하게 다루는 경우가 많고,
# 응답이 안정 영역으로 들어가는 전체 과도 구간 길이를 보는 데 사용
print("TransientTime:", eng.eval("info_pos.TransientTime", nargout=1))

# SettlingTime:
# 응답이 최종값 주변 허용 오차 범위 안에 들어간 뒤
# 다시 벗어나지 않고 계속 머무르게 되는 시간
print("SettlingTime:", eng.eval("info_pos.SettlingTime", nargout=1))

# SettlingMin:
# settling 구간을 판단하는 동안 나타난 최소 응답값
# 과도응답 중 아래쪽으로 얼마나 내려갔는지 볼 때 참고 가능
print("SettlingMin:", eng.eval("info_pos.SettlingMin", nargout=1))

# SettlingMax:
# settling 구간을 판단하는 동안 나타난 최대 응답값
# 보통 peak와 같거나 매우 유사할 수 있음
print("SettlingMax:", eng.eval("info_pos.SettlingMax", nargout=1))

# Overshoot:
# 최종값 대비 얼마나 초과했는지를 %로 나타낸 값
# 예: 7.7이면 목표값보다 최대 7.7% 더 올라갔다는 뜻
print("Overshoot:", eng.eval("info_pos.Overshoot", nargout=1))

# Undershoot:
# 최종값 기준 반대 방향으로 얼마나 더 내려갔는지를 %로 나타낸 값
# 현재 응답에서는 사실상 없어서 -0.0으로 보임
print("Undershoot:", eng.eval("info_pos.Undershoot", nargout=1))

# Peak:
# 응답이 기록한 최대값
# 실제 출력이 가장 높았던 값
print("Peak:", eng.eval("info_pos.Peak", nargout=1))

# PeakTime:
# 최대값 Peak에 도달한 시간
# 첫 번째 overshoot가 언제 발생했는지 보는 핵심 지표
print("PeakTime:", eng.eval("info_pos.PeakTime", nargout=1))

RiseTime: 0.46223587287315143
TransientTime: 3.941383441106845
SettlingTime: 3.941383441106845
SettlingMin: 0.8141643016908532
SettlingMax: 1.076869467168402
Overshoot: 7.705394514974873
Undershoot: -0.0
Peak: 1.076869467168402
PeakTime: 0.8392230080215957


In [17]:
# -------------------------
# 1. 파라미터 정리
# -------------------------
params = {
    "Kp": 10.0,
    "Ki": 10.0,
    "Kd": 1.0,
    "m": 1.0,
    "c": 2.0,
    "k": 5.0,
}

# MATLAB workspace에도 반영
eng.workspace["Kp"] = params["Kp"]
eng.workspace["Ki"] = params["Ki"]
eng.workspace["Kd"] = params["Kd"]
eng.workspace["m"] = params["m"]
eng.workspace["c"] = params["c"]
eng.workspace["k"] = params["k"]

# -------------------------
# 2. 시뮬레이션 실행
# -------------------------
eng.eval(f"out = sim('{model_path}');", nargout=0)

# -------------------------
# 3. step info 계산
# -------------------------
eng.eval("info_pos = stepinfo(out.pos_out(:), out.t_out(:));", nargout=0)

position_metrics = {
    "RiseTime": eng.eval("info_pos.RiseTime", nargout=1),
    "TransientTime": eng.eval("info_pos.TransientTime", nargout=1),
    "SettlingTime": eng.eval("info_pos.SettlingTime", nargout=1),
    "SettlingMin": eng.eval("info_pos.SettlingMin", nargout=1),
    "SettlingMax": eng.eval("info_pos.SettlingMax", nargout=1),
    "Overshoot": eng.eval("info_pos.Overshoot", nargout=1),
    "Undershoot": eng.eval("info_pos.Undershoot", nargout=1),
    "Peak": eng.eval("info_pos.Peak", nargout=1),
    "PeakTime": eng.eval("info_pos.PeakTime", nargout=1),
}

# -------------------------
# 4. 최종 JSON 구조
# -------------------------
result = {
    "params": params,
    "position_metrics": position_metrics,
}

# -------------------------
# 5. 저장
# -------------------------
save_path = Path("msd_result.json")

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"Saved to: {save_path.resolve()}")
print(result)

Saved to: C:\Users\Jaewook\Desktop\Myfolder\프로젝트\Control\msd_result.json
{'params': {'Kp': 10.0, 'Ki': 10.0, 'Kd': 1.0, 'm': 1.0, 'c': 2.0, 'k': 5.0}, 'position_metrics': {'RiseTime': 0.46223587287315143, 'TransientTime': 3.941383441106845, 'SettlingTime': 3.941383441106845, 'SettlingMin': 0.8141643016908532, 'SettlingMax': 1.076869467168402, 'Overshoot': 7.705394514974873, 'Undershoot': -0.0, 'Peak': 1.076869467168402, 'PeakTime': 0.8392230080215957}}
